<a href="https://colab.research.google.com/github/madhumitha006/Madhumitha-Codeboosters-internship-2026/blob/main/day_3_ETL_Pandas_APIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
!pip install requests --quiet
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('ALl libraries imported successfully!')
print(f'pandas : {pd.__version__}')
print(f'request: {requests.__version__}')

ALl libraries imported successfully!
pandas : 2.2.2
request: 2.32.4


In [40]:
raw_df = pd.read_csv('messy_sales_data.csv')
print(f"Dataset loaded: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")
print(f"Columns: {raw_df.columns.tolist()}")
print("\nFirst 3 rows:")
print(raw_df.head(3))


Dataset loaded: 30 rows, 9 columns
Columns: ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']

First 3 rows:
   order_id customer_name   product     category  quantity  unit_price  \
0      1001  Ramesh Kumar    Laptop  Electronics       2.0       45000   
1      1002    Priya Nair       NaN  Electronics       1.0       15000   
2      1003    AMIT VERMA  Keyboard  Accessories       3.0        1200   

   order_date       city    sales_rep  
0  2024-01-05     Mumbai  Anil Sharma  
1  2024-01-07      Delhi   Sunita Rao  
2  2024-01-08  Bangalore  Anil Sharma  


In [41]:
print('=' *55)
print(' DATA QUALITY DIAGNOSIS REPORT')
print('=' *55)

print('\n[1] Missing Values per columns')
print(raw_df.isnull().sum())

print(f'\n[2] DUPLICATE ROWS: {raw_df.duplicated().sum()}')
print('\n[3] DATA TYPES:')
print(raw_df.dtypes)

print('\n[4] UNIQUE CATEGORIES:',raw_df['category'].unique())
print('\n[4] Sample customer names:',raw_df['customer_name'].dropna().unique()[:8])
print('\n[4] Sample order_date values:',raw_df['order_date'].unique()[:6])

 DATA QUALITY DIAGNOSIS REPORT

[1] Missing Values per columns
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] DUPLICATE ROWS: 0

[3] DATA TYPES:
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] UNIQUE CATEGORIES: ['Electronics' 'Accessories' nan]

[4] Sample customer names: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh' 'Ananya Das' 'Vikram Iyer']

[4] Sample order_date values: ['2024-01-05' '2024-01-07' '2024-01-08' '2024-01-10' '07-01-2024'
 '2024-01-12']


In [42]:
df = raw_df.copy()
print(f'Working copy created: {df.shape}')
print('raw_of is untouched - we can always react by running df = raw_of_copy()')

Working copy created: (30, 9)
raw_of is untouched - we can always react by running df = raw_of_copy()


In [43]:
print('Before fixing nulls:',df.isnull().sum().sum(),'total existing values')
median_qty = df['quantity'].median()
df['quantity'].fillna(median_qty, inplace=True)
print(f"Filled missing quantity with median: {median_qty}")
df['category'].fillna('Uncategorized', inplace=True)
print("After fixing nulls:", df.isnull().sum().sum(), "total missing values")

Before fixing nulls: 7 total existing values
Filled missing quantity with median: 2.0
After fixing nulls: 3 total missing values


In [44]:
print(f'Before duplication: {len(df)} rows')
print(f'Duplicate rows: {df.duplicated().sum()}')
print('\nDuplicate rows:')
print(df[df.duplicated(keep=False)][['order_id', 'customer_name', 'product', 'order_date']])
df.drop_duplicates(inplace=True)
print(f'After removing duplicates: {len(df)} rows')
print(f'Duplicate rows: {df.duplicated().sum()}')


Before duplication: 30 rows
Duplicate rows: 0

Duplicate rows:
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []
After removing duplicates: 30 rows
Duplicate rows: 0


In [45]:
print('Sample dates before parsing:')
print(df['order_date'].head(8).tolist())
df['order_date'] = pd.to_datetime(
    df['order_date'],
    dayfirst=False,
    errors='coerce'
)
nat_count = df['order_date'].isnull().sum()
print(f'\nUnparseable dates (NaT): {nat_count}')
df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['monthname'] = df['order_date'].dt.strftime('%B')
print('\nSample dates after parsing:')
print(df[['order_date', 'year', 'month', 'monthname']].head(8))


Sample dates before parsing:
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13']

Unparseable dates (NaT): 2

Sample dates after parsing:
  order_date    year  month monthname
0 2024-01-05  2024.0    1.0   January
1 2024-01-07  2024.0    1.0   January
2 2024-01-08  2024.0    1.0   January
3 2024-01-10  2024.0    1.0   January
4 2024-01-05  2024.0    1.0   January
5        NaT     NaN    NaN       NaN
6 2024-01-12  2024.0    1.0   January
7 2024-01-13  2024.0    1.0   January


In [46]:
print('Before standardization:', df['customer_name'].unique()[:6])
df['customer_name'] = (
    df['customer_name']
    .str.strip()
    .str.title()
)
print('After standardization:', df['customer_name'].unique()[:6])
print('\nBefore: Keyboard rows with Electronics category:')
wrong_mask = (df['product'] == 'Keyboard') & (df['category'] == 'Electronics')
print(df[wrong_mask][['product', 'category']])
df.loc[wrong_mask, 'category'] = 'Accessories'
print('After fix: Unique categories:', df['category'].unique())

Before standardization: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh']
After standardization: ['Ramesh Kumar' 'Priya Nair' 'Amit Verma' 'Sunita Patel' 'Kiran Mehta'
 'Deepak Singh']

Before: Keyboard rows with Electronics category:
     product     category
23  Keyboard  Electronics
After fix: Unique categories: ['Electronics' 'Accessories' 'Uncategorized']


In [47]:
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce').astype(int)
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
df['revenue'] = df['quantity'] * df['unit_price']
print('revenue column created:')
print(df[['customer_name', 'product', 'quantity', 'revenue']].head(5))
print(f"\nTotal revenue: ${df['revenue'].sum():.0f}")

revenue column created:
  customer_name   product  quantity  revenue
0  Ramesh Kumar    Laptop         2    90000
1    Priya Nair       NaN         1    15000
2    Amit Verma  Keyboard         3     3600
3  Sunita Patel   Monitor         2    44000
4  Ramesh Kumar    Laptop         2    90000

Total revenue: $818000


In [48]:
print('=' * 55)
print('POST-CLEANING VALIDATION REPORT')
print('=' * 55)
print(f'Original rows  : {len(raw_df)}')
print(f'Cleaned rows   : {len(df)}')
print(f'Rows removed   : {len(raw_df) - len(df)} (duplicates)')
print(f'Missing values : {df.isnull().sum().sum()}')
print(f'Duplicate      : {df.duplicated().sum()}')
print(f'Date nulls     : {df["order_date"].isnull().sum()}')
print(f'Revenue NaN    : {df["revenue"].isnull().sum()}')
print(f'Categories     : {sorted(df["category"].unique())}')
all_clean = (
    df.isnull().sum().sum() == 0 and
    df.duplicated().sum() == 0
)
print(f'DATA IS CLEAN: {all_clean}')

POST-CLEANING VALIDATION REPORT
Original rows  : 30
Cleaned rows   : 30
Rows removed   : 0 (duplicates)
Missing values : 11
Duplicate      : 0
Date nulls     : 2
Revenue NaN    : 0
Categories     : ['Accessories', 'Electronics', 'Uncategorized']
DATA IS CLEAN: False


In [49]:
product_rev = (
    df.groupby('product')['revenue']
    .sum()
    .reset_index()
    .sort_values(by='revenue', ascending=False)
)
print('Revenue by Product:')
print(product_rev.to_string(index=False))
category_summary = (df.groupby('category').agg(
        total_revenue=('revenue', 'sum'),
        average_order_value=('revenue', 'mean'),
        num_orders=('order_id', 'nunique'),
        unique_products=('product', 'nunique')
    ).round(2).reset_index())
print('\nCategory Summary:')
print(category_summary.to_string(index=False))

Revenue by Product:
   product  revenue
    Laptop   540000
   Monitor   154000
Headphones    28000
     Mouse    20800
  Keyboard    20400
    Webcam    20000
   USB Hub    19800

Category Summary:
     category  total_revenue  average_order_value  num_orders  unique_products
  Accessories          81000              5785.71          14                4
  Electronics         693000             46200.00          15                3
Uncategorized          44000             44000.00           1                1


In [50]:
print("Cleaned data saved to clean_sales_data.csv")
print(f"Final dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nETL Pipeline for Sales Data (OUTPUT)")
print("EXTRACT -> messy_sales_data.csv loaded")
print("TRANSFORM -> null values fixed, duplicates removed, data parsed")
print("LOAD -> clean_sales_data.csv saved")

Cleaned data saved to clean_sales_data.csv
Final dataset: 30 rows, 13 columns

ETL Pipeline for Sales Data (OUTPUT)
EXTRACT -> messy_sales_data.csv loaded
TRANSFORM -> null values fixed, duplicates removed, data parsed
LOAD -> clean_sales_data.csv saved


Practice Questions

Q1: What are the three stages of ETL? Describe each stage using an example from today's sales dataset.

Q2: A DataFrame has 500 rows. After calling df.dropna(), it has 412 rows. What does this tell you?

Q3: Write code to remove duplicates from df where "same row" means same customer_name AND same product.

Q4: What is the difference between fillna(0) and fillna(df['col'].median())? When would you prefer each?

Q5: Write Python code to call the weather API for "Delhi" and print the temperature in Celsius.

Q6: What does response.status_code == 200 mean? What should you do when the code is 401?

Q1.
ETL stands for Extract, Transform, Load.

1.Extract – Data is collected from a source such as CSV files, APIs, or databases.
Example: Load messy_sales_data.csv.

2.Transform – Data is cleaned and processed.
Example: Remove duplicates, fill null values, convert data types.

3.Load – Store the cleaned data into another file or database.
Example: Save cleaned data as clean_sales_data.csv.

In [55]:
# Q1 - ETL Process

import pandas as pd

# Extract
df = pd.read_csv("messy_sales_data.csv")

# Transform
df = df.drop_duplicates()
df = df.fillna(0)

# Load
df.to_csv("clean_sales_data.csv", index=False)

In [56]:
# Q2 - Rows removed after dropna()

total_rows = 500
remaining_rows = 412

removed = total_rows - remaining_rows

print("Rows removed:", removed)
print("Rows with missing values:", removed)

Rows removed: 88
Rows with missing values: 88


In [57]:
# Q3 - Remove duplicate customer and product rows

df = df.drop_duplicates(
    subset=['customer_name', 'product']
)

print(df)

    order_id  customer_name     product     category  quantity  unit_price  \
0       1001   Ramesh Kumar      Laptop  Electronics       2.0       45000   
1       1002     Priya Nair           0  Electronics       1.0       15000   
2       1003     AMIT VERMA    Keyboard  Accessories       3.0        1200   
3       1004   Sunita Patel     Monitor  Electronics       0.0       22000   
5       1006    kiran mehta       Mouse  Accessories      10.0         800   
6       1007   Deepak Singh  Headphones  Electronics       2.0        3500   
7       1008              0      Webcam  Accessories       1.0        2500   
8       1009     Ananya Das      Laptop  Electronics       1.0       45000   
9       1010    Vikram Iyer    Keyboard  Accessories       5.0        1200   
10      1011    Pooja Gupta     Monitor  Electronics       2.0       22000   
11      1012     SURESH RAO     USB Hub  Accessories       8.0         600   
12      1013    Meera Joshi      Laptop  Electronics       0.0  

In [59]:
#Q4- fillna examples
df['unit_price'] = df['unit_price'].fillna(
    df['unit_price'].median()
)

print(df.head())

   order_id customer_name   product     category  quantity  unit_price  \
0      1001  Ramesh Kumar    Laptop  Electronics       2.0       45000   
1      1002    Priya Nair         0  Electronics       1.0       15000   
2      1003    AMIT VERMA  Keyboard  Accessories       3.0        1200   
3      1004  Sunita Patel   Monitor  Electronics       0.0       22000   
5      1006   kiran mehta     Mouse  Accessories      10.0         800   

   order_date       city    sales_rep  
0  2024-01-05     Mumbai  Anil Sharma  
1  2024-01-07      Delhi   Sunita Rao  
2  2024-01-08  Bangalore  Anil Sharma  
3  2024-01-10    Chennai   Ravi Kumar  
5  07-01-2024       Pune   Sunita Rao  


In [61]:
# Q5 - Weather API for Delhi

import requests

city = "Delhi"
api_key = "YOUR_API_KEY"

url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"

response = requests.get(url)

data = response.json()

if response.status_code == 200:
    print("Temperature:", data['main']['temp'], "°C")
else:
    print("Error:", data.get('message', 'Unable to fetch weather'))

Error: Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.
